# AI for Handwriting Identification — Colab quickstart

**Colab has no `C:\Users\...`.** On your PC the release may live at `C:\Users\thisb\Downloads\AnyScriptFiltered.tar.gz`. For Colab: **upload that `.tar.gz` to Google Drive** (e.g. `MyDrive/`), then use a **`/content/drive/MyDrive/...`** path below and extract there so training survives disconnects.

**Dataset:** `DATA_ROOT` = folder whose **children are author IDs** (default `/content/drive/MyDrive/AnyScriptFiltered`). Layouts: `author/book/*.jpg` or `author/*.png`.

## Part A, B, C (run via `/content/ai-hw/scripts/...`; shell stays in `/content` so `pip` never breaks after `rm -rf`)

| Part | Script | Output |
|------|--------|--------|
| **A — Train** | `train_triplet_unsloth.py` | `OUT/best.pt` |
| **B — FAISS** | `build_faiss_index.py` | `OUT/faiss.index`, `OUT/meta.npy` |
| **C — Submit CSV** | `export_anyscript_submission.py` | dense `intra_book` / `extra_book` CSV |

More detail: **`README.md`** (Quick start + ICDAR submission).

In [ ]:
# Optional: persist data & checkpoints on Drive
# If mount hangs, this forces a clean remount.
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass

drive.mount("/content/drive", force_remount=True)

### Reset working directory (run if you see `getcwd` / `Unable to read current working directory`)

If your shell was inside `/content/ai-hw` and that folder was deleted, **run the next cell** before **Clone** or **`git pull`**.

In [ ]:
# Clone repo (fork/URL ok — change if you renamed the repo)
# Do NOT add %cd /content/ai-hw/scripts — re-run + rm deletes cwd (getcwd / git clone fail).
%cd /content
import os
os.chdir("/content")
!rm -rf /content/ai-hw
!git clone https://github.com/Todd838/AI-for-Handwriting-Identification.git /content/ai-hw
# Do not %cd into the repo: if you re-run this cell, rm -rf removes your cwd and pip breaks.
print("Repo:", "/content/ai-hw/scripts — run scripts with full path below.")

### Update code from GitHub (optional)

If **`/content/ai-hw` already exists** from an earlier run and you only want the latest `main` branch, run the **next cell** instead of re-running **Clone**. **Skip** if you just ran Clone. If you see `fatal: not a git repository`, run **Clone** first.

In [ ]:
%cd /content
!cd /content/ai-hw && git pull origin main

In [ ]:
# Install everything from the repo (same as local requirements.txt)
# Safe on fresh runtimes: auto-clones repo if /content/ai-hw is missing.
import os
import subprocess

os.chdir("/content")
repo_dir = "/content/ai-hw"
req_path = f"{repo_dir}/requirements.txt"
if not os.path.isfile(req_path):
    print("/content/ai-hw missing; cloning fresh repo...")
    subprocess.run(["rm", "-rf", repo_dir], check=False)
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Todd838/AI-for-Handwriting-Identification.git",
            repo_dir,
        ],
        check=True,
    )

%pip install -q -r /content/ai-hw/requirements.txt
# If pip errors on a package, run: %pip install -q unsloth  (after GPU runtime is on)

In [ ]:
# Unsloth (use Runtime → GPU first). If install fails, train with --no_unsloth
import os
os.chdir("/content")
import torch
print("CUDA available:", torch.cuda.is_available())
%pip install -q unsloth

### GPU required for Unsloth

**`CUDA available: False`** means this runtime has **no GPU**. Use **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or L4 / A100 if offered), then **Runtime → Restart runtime** and **re-run** cells from the top (mount → clone → install → this cell).

Without a GPU you can still try training with **`--no_unsloth`** (much slower / may OOM on large models).

In [ ]:
# Dataset on Colab = Drive path, NOT C:\Users\thisb\Downloads\...
# Logic is in /content/ai-hw/scripts/colab_dataset_setup.py so `git pull` fixes imports
# even when this Colab tab still shows an old copy of the notebook.
import os
import sys

os.chdir("/content")
_setup = "/content/ai-hw/scripts/colab_dataset_setup.py"
if not os.path.isfile(_setup):
    raise FileNotFoundError(
        _setup
        + " missing. Run Clone, then: !cd /content/ai-hw && git pull origin main"
    )
_g = globals()
exec(compile(open(_setup, encoding="utf-8").read(), _setup, "exec"), _g, _g)

## Part A — training

Re-run the **dataset** cell above **in this session** before training so `DATA_ROOT` is set (it must not stay the empty `AnyScriptFiltered` folder). Then run the next cell. Checkpoints go to **`OUT`** on Drive.

### If Part A says `0 pages found`

1. If `AnyScriptFiltered/` only has **`.tar.gz`**, you must **extract** first (the dataset cell now tries both `MyDrive/AnyScriptFiltered.tar.gz` and `MyDrive/AnyScriptFiltered/AnyScriptFiltered.tar.gz`).
2. Use **`--data_root auto`** in the train command so path resolution runs inside `train_triplet_unsloth.py`.
3. **Do not** use `DATA_ROOT` from unrelated projects (e.g. other Kaggle folders). The inspector only suggests **AnyScript-related** paths unless you pass `--force-unrelated-scan`.

### Shell `!python` vs `{OUT}` / `{DATA_ROOT}`

Lines starting with **`!`** are run by the **shell**, not Python: **`{OUT}` and `{DATA_ROOT}` are not expanded** (you get a literal path like `{OUT}/best.pt` and `FileNotFoundError`). Use the **Part B** cell below (`subprocess` + real paths), or in `!python` use **`--data_root auto`** (same as training) and paste the full checkpoint path.

In [ ]:
import os
import subprocess
import sys

repo_dir = "/content/ai-hw"
script_path = f"{repo_dir}/scripts/inspect_anyscript_layout.py"

# Finds which subfolder actually contains page images.
# If /content/ai-hw is missing in this runtime, auto-clone it first.
# Run the Drive mount cell first — if every path shows exists=False, Drive is not mounted.
%cd /content

if not os.path.isfile(script_path):
    print("/content/ai-hw missing; cloning fresh repo...")
    subprocess.run(["rm", "-rf", repo_dir], check=False)
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Todd838/AI-for-Handwriting-Identification.git",
            repo_dir,
        ],
        check=True,
    )

try:
    # Running without --data_root auto, as exit code 2 suggests an invalid argument
    result = subprocess.run([sys.executable, script_path], capture_output=True, text=True, check=True)
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Command failed with exit status {e.returncode}.")
    print("\n--- Standard Error Output ---")
    print(e.stderr)
    print("--- Standard Output ---")
    print(e.stdout)

In [ ]:
OUT = "/content/drive/MyDrive/anyscript_runs/run1"
import os
import subprocess
import sys

os.environ["ANYSCRIPT_OUT"] = OUT
os.makedirs(OUT, exist_ok=True)
os.chdir("/content")

base = [
    sys.executable,
    "/content/ai-hw/scripts/train_triplet_unsloth.py",
    "--data_root",
    "auto",
    "--model_name",
    "zai-org/GLM-OCR",
    "--output_dir",
    OUT,
    "--epochs",
    "3",
    "--steps_per_epoch",
    "2000",
]

attempts = [
    ("batch_size=4 (preferred)", ["--batch_size", "4"]),
    ("batch_size=2 (VRAM fallback)", ["--batch_size", "2"]),
    ("batch_size=2 + --no_unsloth (compat fallback)", ["--batch_size", "2", "--no_unsloth"]),
]

last_err = None
for label, extra in attempts:
    cmd = base + extra
    print(f"\n=== Training attempt: {label} ===")
    try:
        subprocess.run(cmd, check=True)
        print("Training completed successfully.")
        last_err = None
        break
    except subprocess.CalledProcessError as e:
        last_err = e
        print(f"Attempt failed with exit code {e.returncode}. Trying next fallback...")

if last_err is not None:
    raise RuntimeError(
        "All training attempts failed. Scroll up to the first real error message from "
        "train_triplet_unsloth.py and paste it here for a targeted fix."
    )

## Part B — FAISS index

Run after **`best.pt`** exists under `OUT`, then run the **next cell** (it uses `subprocess` + `--data_root auto`, not shell `!python` with `{OUT}` — those braces are not expanded in `!` lines).

The command enables `--resume` + periodic checkpoints, so reconnecting Colab and re-running Part B continues from saved progress.

If your notebook still shows an old commented `!python ... {DATA_ROOT} ...` Part B, open **`/content/ai-hw/colab_quickstart.ipynb`** after **Clone / git pull**, or re-open the notebook from GitHub.

In [ ]:
# Part B — full-gallery FAISS with resume checkpoints.
# Safe to re-run after disconnect: it resumes from existing faiss.index/meta.npy.
# batch_size=1 processes one page at a time to reduce memory pressure and crashes.
import os
import subprocess
import sys

if "OUT" not in globals() or not OUT:
    raise RuntimeError("Set OUT in the Part A cell above (same path as --output_dir).")
_ckpt = os.path.join(OUT, "best.pt")
if not os.path.isfile(_ckpt):
    raise FileNotFoundError(_ckpt + " — train Part A first or fix OUT.")

subprocess.run(
    [
        sys.executable,
        "/content/ai-hw/scripts/build_faiss_index.py",
        "--data_root",
        "auto",
        "--checkpoint",
        _ckpt,
        "--index_out",
        os.path.join(OUT, "faiss.index"),
        "--meta_out",
        os.path.join(OUT, "meta.npy"),
        "--batch_size",
        "1",
        "--all_pages",
        "--resume",
        "--save_every_batches",
        "25",
    ],
    check=True,
)

## Part C — submission CSV

Set **`QUERY_ROOT`** to held-out query images, then run the next cell. **`GALLERY_ROOT`** is usually **`DATA_ROOT`**. For a real upload, set **`QUERY_IDS_JSON`** and **`GALLERY_IDS_JSON`** too (see **`README.md`**). Use synthetic IDs only for local testing.

In [ ]:
QUERY_ROOT = "/content/drive/MyDrive/anyscript_query_holdout"
GALLERY_ROOT = DATA_ROOT
QUERY_IDS_JSON = None  # e.g. "/content/drive/MyDrive/query_ids.json" for real submission
GALLERY_IDS_JSON = None  # e.g. "/content/drive/MyDrive/gallery_ids.json" for real submission
ALLOW_SYNTHETIC_IDS = True  # set False for real competition runs

import os
import subprocess
import sys

if "OUT" not in globals() or not OUT:
    raise RuntimeError("Set OUT in Part A cell first.")
if not QUERY_ROOT or not os.path.isdir(QUERY_ROOT):
    raise RuntimeError(f"Set QUERY_ROOT to an existing folder, got: {QUERY_ROOT!r}")
if not GALLERY_ROOT or not os.path.isdir(GALLERY_ROOT):
    raise RuntimeError(f"Set GALLERY_ROOT to an existing folder, got: {GALLERY_ROOT!r}")

os.chdir("/content")
_ckpt = os.path.join(OUT, "best.pt")
if not os.path.isfile(_ckpt):
    raise FileNotFoundError(_ckpt + " — train Part A first or fix OUT.")

def run_submission(mode: str, out_name: str):
    cmd = [
        sys.executable,
        "/content/ai-hw/scripts/export_anyscript_submission.py",
        "--mode",
        mode,
        "--checkpoint",
        _ckpt,
        "--gallery_data_root",
        GALLERY_ROOT,
        "--query_data_root",
        QUERY_ROOT,
        "--out_csv",
        os.path.join(OUT, out_name),
        "--batch_size",
        "1",
        "--query_chunk",
        "1",
    ]
    if ALLOW_SYNTHETIC_IDS:
        cmd.append("--allow_synthetic_ids")
    if QUERY_IDS_JSON:
        cmd.extend(["--query_ids_json", QUERY_IDS_JSON])
    if GALLERY_IDS_JSON:
        cmd.extend(["--gallery_ids_json", GALLERY_IDS_JSON])
    subprocess.run(cmd, check=True)

run_submission("intra_book", "submission_intra_book.csv")
run_submission("extra_book", "submission_extra_book.csv")